In [ ]:
import numpy as np
import time
import pandas as pd
from dataclasses import dataclass
from typing import Callable, List, Literal, Tuple

# ==========================
# 1. Synthetic data + utilities
# ==========================

def generate_synthetic_data(
    n: int = 1000,
    d: int = 4,
    mode: str = "independent",
    seed: int = 0
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    if mode == "independent":
        X = rng.random((n, d))
    elif mode == "anti_correlated":
        base = rng.random((n, 1))
        noise = 0.05 * rng.standard_normal((n, d - 1))
        others = 1.0 - base + noise
        X = np.concatenate([base, others], axis=1)
        X = np.clip(X, 0.0, 1.0)
    elif mode == "clustered":
        k = min(5, n)
        centers = rng.random((k, d))
        assignments = rng.integers(0, k, size=n)
        X = centers[assignments] + 0.05 * rng.standard_normal((n, d))
        X = np.clip(X, 0.0, 1.0)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return X


@dataclass
class PowerUtility:
    '''
    Nonlinear power utility:
      f(x) = sum_i w_i * x_i^{m_i}, with w on the simplex and m_i > 0.
    '''
    w: np.ndarray
    m: np.ndarray

    def __post_init__(self):
        self.w = np.asarray(self.w, dtype=float)
        self.m = np.asarray(self.m, dtype=float)
        if self.w.ndim != 1 or self.m.ndim != 1:
            raise ValueError("w and m must be 1D")
        if self.w.shape[0] != self.m.shape[0]:
            raise ValueError("w and m must have same length")
        if np.any(self.w < 0):
            raise ValueError("weights must be nonnegative")
        s = self.w.sum()
        if s <= 0:
            raise ValueError("sum of weights must be > 0")
        self.w = self.w / s
        if np.any(self.m <= 0):
            raise ValueError("exponents m_i must be > 0")

    @classmethod
    def random(
        cls,
        d: int,
        rng: np.random.Generator = None,
        m_low: float = 0.3,
        m_high: float = 2.0
    ) -> "PowerUtility":
        if rng is None:
            rng = np.random.default_rng()
        w = rng.random(d)
        w = w / w.sum()
        m = rng.uniform(m_low, m_high, size=d)
        return cls(w=w, m=m)

    def __call__(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.w.shape[0]:
            raise ValueError("X must be (n,d)")
        Xc = np.clip(X, 1e-9, 1.0)
        return (self.w * (Xc ** self.m)).sum(axis=1)


@dataclass
class LinearUtility:
    '''
    Linear surrogate using only coefficients (weights) — no exponents.
    '''
    u: np.ndarray

    def __post_init__(self):
        self.u = np.asarray(self.u, dtype=float)
        if self.u.ndim != 1:
            raise ValueError("u must be 1D")
        if np.any(self.u < 0):
            raise ValueError("u must be nonnegative")
        s = self.u.sum()
        if s <= 0:
            raise ValueError("sum of u must be > 0")
        self.u = self.u / s

    def __call__(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        return X @ self.u


@dataclass
class UserOracleNonlinear:
    '''
    Real user: chooses with nonlinear PowerUtility.
    '''
    utility: PowerUtility

    def choose(self, tuples: np.ndarray) -> int:
        vals = self.utility(tuples)
        return int(np.argmax(vals))


# ==========================
# 2. Helpers: skyline, nonlinear IQ, regret
# ==========================

def skyline_indices(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        for j in range(n):
            if i == j or dominated[j]:
                continue
            if np.all(X[j] >= X[i]) and np.any(X[j] > X[i]):
                dominated[i] = True
                break
    return np.where(~dominated)[0]


def epsilon_skyline_indices(X: np.ndarray, c: float) -> np.ndarray:
    '''
    (1+eps)-skyline: p c-dominates q if
      p >= c * q componentwise and strictly greater in some dimension.
    '''
    if c < 1.0:
        raise ValueError("c must be >= 1")
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    cdom = np.zeros(n, dtype=bool)
    for i in range(n):
        if cdom[i]:
            continue
        for j in range(n):
            if i == j or cdom[j]:
                continue
            if np.all(X[j] >= c * X[i]) and np.any(X[j] > c * X[i]):
                cdom[i] = True
                break
    return np.where(~cdom)[0]


def top_k_indices(
    X: np.ndarray,
    utility: Callable[[np.ndarray], np.ndarray],
    k: int
) -> np.ndarray:
    vals = utility(X)
    if k >= len(vals):
        return np.argsort(-vals)
    return np.argpartition(-vals, k - 1)[:k]


def nonlinear_IQ_indices(
    X: np.ndarray,
    utility: PowerUtility,
    eps: float
) -> np.ndarray:
    '''
    True indistinguishability set for nonlinear utility f:
      I = {p: f(p*) / f(p) <= 1 + eps}.
    '''
    vals = utility(X)
    best_idx = int(np.argmax(vals))
    best_val = vals[best_idx]
    thr = best_val / (1.0 + eps)
    return np.where(vals >= thr)[0]


def regret_ratio_nonlinear(
    X: np.ndarray,
    S_idx: np.ndarray,
    utility: PowerUtility
) -> float:
    vals = utility(X)
    best_val = float(vals.max())
    if best_val <= 0:
        return 0.0
    if len(S_idx) == 0:
        return 1.0
    best_in_S = float(vals[S_idx].max())
    return (best_val - best_in_S) / best_val


# ==========================
# 3. Squeeze-u (linear IQ algorithm; user is nonlinear)
# ==========================

@dataclass
class SqueezeUResult:
    L: np.ndarray
    H: np.ndarray
    chosen_dim: int
    questions_used: int
    output_indices: np.ndarray


def squeeze_u_linear_on_nonlinear_user(
    X: np.ndarray,
    oracle: UserOracleNonlinear,
    eps: float,
    s: int = 2,
    q_max: int = 20
) -> SqueezeUResult:
    '''
    Squeeze-u as in IQ (linear), but the user answers with nonlinear utility.
    '''
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    if s < 2:
        raise ValueError("s must be >= 2")
    q_used = 0

    # (1+eps)-skyline pre-prune
    idx_csky = epsilon_skyline_indices(X, c=1.0 + eps)
    D = X[idx_csky]
    idx_map = idx_csky

    # per-dim min/max
    m = D.min(axis=0)
    M = D.max(axis=0)

    # artificial e_i
    E = np.empty((d, d), dtype=float)
    for i in range(d):
        ei = m.copy()
        ei[i] = m[i] + (M[i] - m[i]) / 2.0
        E[i] = ei

    # Phase 1: find i*
    i_star = 0
    i = 1
    while i < d and q_used < q_max:
        group_dims = [i_star] + list(range(i, min(d, i + (s - 1))))
        group = E[group_dims]
        choice_idx = oracle.choose(group)
        i_star = group_dims[choice_idx]
        i += (s - 1)
        q_used += 1

    # Initialize L,H
    L = np.zeros(d, dtype=float)
    H = np.ones(d, dtype=float)
    L[i_star] = 1.0
    H[i_star] = 1.0

    # Phase 2: refine bounds
    current_dim = 0
    while q_used < q_max:
        if current_dim == i_star:
            current_dim = (current_dim + 1) % d
            continue
        i = current_dim
        # φ grid
        phi = np.array([L[i] + j * (H[i] - L[i]) / s for j in range(s + 1)])
        # build P
        P = np.zeros((s, d), dtype=float)
        for k in range(1, s + 1):
            idx = k - 1
            if k <= s - 1:
                P[idx, i_star] = phi[k: s].sum() / s
            else:
                P[idx, i_star] = 0.0
            P[idx, i] = k / s
        choice_idx = oracle.choose(P)
        c = choice_idx + 1
        L[i] = phi[c - 1]
        H[i] = phi[c]
        q_used += 1
        current_dim = (current_dim + 1) % d

    # Final pruning (box [L,H] heuristic on D)
    V_vals = D @ L
    V = float(V_vals.max())
    keep_mask = (1.0 + eps) * (D @ H) >= V
    out_local = np.where(keep_mask)[0]
    out_idx = idx_map[out_local]

    return SqueezeUResult(L=L, H=H, chosen_dim=i_star,
                          questions_used=q_used, output_indices=out_idx)


# ==========================
# 4. MinR / MinD with Monte Carlo feasible region
#    (linear assumption, nonlinear user)
# ==========================

@dataclass
class FeasibleRegionConstraints:
    d: int
    constraints: List[np.ndarray]

    @classmethod
    def initial(cls, d: int) -> "FeasibleRegionConstraints":
        return cls(d=d, constraints=[])

    def add_comparison(self, c: np.ndarray, p: np.ndarray) -> None:
        diff = np.asarray(c - p, dtype=float)
        if diff.shape != (self.d,):
            raise ValueError("c,p must be (d,)")
        self.constraints.append(diff)


def sample_feasible_utilities(
    region: FeasibleRegionConstraints,
    num_samples: int,
    max_tries: int = None,
    rng: np.random.Generator = None
) -> np.ndarray:
    '''
    Monte Carlo sampler for linear weights u on simplex satisfying
      (c-p)·u >= 0 for all constraints (c,p).
    '''
    if rng is None:
        rng = np.random.default_rng()
    if max_tries is None:
        max_tries = num_samples * 10
    d = region.d
    accepted = []
    tries = 0
    while len(accepted) < num_samples and tries < max_tries:
        u = rng.dirichlet(alpha=np.ones(d))
        ok = True
        for a in region.constraints:
            if a @ u < 0:
                ok = False
                break
        if ok:
            accepted.append(u)
        tries += 1
    if not accepted:
        return np.empty((0, d), dtype=float)
    return np.vstack(accepted)


def approx_width_from_samples(samples: np.ndarray) -> float:
    if samples.size == 0:
        return float("inf")
    L = samples.min(axis=0)
    H = samples.max(axis=0)
    return float(np.max(H - L))


def approx_diameter_from_samples(samples: np.ndarray) -> float:
    if samples.size == 0:
        return float("inf")
    k = samples.shape[0]
    max_d2 = 0.0
    for i in range(k):
        diff = samples[i+1:] - samples[i]
        if diff.size == 0:
            continue
        d2 = np.sum(diff * diff, axis=1)
        md2 = float(d2.max())
        if md2 > max_d2:
            max_d2 = md2
    return float(np.sqrt(max_d2))


@dataclass
class MinRMinDResult:
    algorithm: Literal["MinR", "MinD"]
    questions_used: int
    region: FeasibleRegionConstraints
    L: np.ndarray
    H: np.ndarray
    output_indices: np.ndarray


def minR_minD_linear_on_nonlinear_user(
    X: np.ndarray,
    oracle: UserOracleNonlinear,
    eps: float,
    s: int = 5,
    T: int = 20,
    q_max: int = 10,
    algo: Literal["MinR", "MinD"] = "MinR",
    num_samples_for_R: int = 200,
    seed: int = 0
) -> MinRMinDResult:
    '''
    MinR / MinD from IQ (linear), but user is nonlinear.
    We keep them LP-free by Monte Carlo approximations.
    '''
    assert algo in ("MinR", "MinD")
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    n, d = X.shape

    # initial C: (1+eps)-skyline
    idx_csky = epsilon_skyline_indices(X, c=1.0 + eps)
    C_idx = idx_csky.copy()

    region = FeasibleRegionConstraints.initial(d=d)
    q_used = 0

    while q_used < q_max and len(C_idx) >= s:
        best_S_idx = None
        best_metric = None

        for _ in range(T):
            if len(C_idx) == s:
                S_idx = C_idx
            else:
                S_idx = rng.choice(C_idx, size=s, replace=False)
            S_idx = np.array(S_idx, dtype=int)

            metrics = []
            for c_idx in S_idx:
                c = X[c_idx]
                tmp_region = FeasibleRegionConstraints(d=d, constraints=list(region.constraints))
                for p_idx in S_idx:
                    if int(p_idx) == int(c_idx):
                        continue
                    p = X[int(p_idx)]
                    tmp_region.add_comparison(c, p)
                samples = sample_feasible_utilities(tmp_region, num_samples_for_R, rng=rng)
                if algo == "MinR":
                    mval = approx_width_from_samples(samples)
                else:
                    mval = approx_diameter_from_samples(samples)
                metrics.append(mval)
            avg_metric = float(np.mean(metrics)) if metrics else float("inf")
            if best_metric is None or avg_metric < best_metric:
                best_metric = avg_metric
                best_S_idx = S_idx

        # ask user
        S = X[best_S_idx]
        choice_local = oracle.choose(S)
        c_idx = int(best_S_idx[choice_local])
        c = X[c_idx]
        for p_idx in best_S_idx:
            if int(p_idx) == c_idx:
                continue
            p = X[int(p_idx)]
            region.add_comparison(c, p)
        q_used += 1

        # prune C_idx via sampled L,H
        samples_global = sample_feasible_utilities(region, num_samples_for_R, rng=rng)
        if samples_global.size == 0:
            continue
        L = samples_global.min(axis=0)
        H = samples_global.max(axis=0)
        C_vals_L = X[C_idx] @ L
        V = float(C_vals_L.max())
        keep_mask = (1.0 + eps) * (X[C_idx] @ H) >= V
        C_idx = C_idx[keep_mask]

    samples_final = sample_feasible_utilities(region, num_samples_for_R,
                                              rng=np.random.default_rng(seed+1))
    if samples_final.size > 0:
        L_final = samples_final.min(axis=0)
        H_final = samples_final.max(axis=0)
    else:
        L_final = np.zeros(d)
        H_final = np.ones(d)

    return MinRMinDResult(
        algorithm=algo,
        questions_used=q_used,
        region=region,
        L=L_final,
        H=H_final,
        output_indices=C_idx
    )


# ==========================
# 5. IOD for nonlinear utilities via Monte Carlo parameter samples
# ==========================

@dataclass
class NonlinearIODMCResult:
    questions_used: int
    num_params_remaining: int
    output_indices: np.ndarray


def sample_power_params(
    d: int,
    num_samples: int,
    rng: np.random.Generator,
    m_low: float = 0.3,
    m_high: float = 2.0
) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Draw num_samples candidate (w, m) parameter vectors.
      w_k on simplex, m_k in [m_low, m_high]^d.
    Returns:
      W: (num_samples, d)
      M: (num_samples, d)
    '''
    W = rng.dirichlet(alpha=np.ones(d), size=num_samples)
    M = rng.uniform(m_low, m_high, size=(num_samples, d))
    return W, M


def iod_param_best_index(
    tuples: np.ndarray,
    w: np.ndarray,
    m: np.ndarray
) -> int:
    '''
    For a single (w,m), return argmax index in 'tuples'.
    '''
    util = PowerUtility(w=w, m=m)
    vals = util(tuples)
    return int(np.argmax(vals))


def update_param_samples_with_query(
    param_W: np.ndarray,
    param_M: np.ndarray,
    tuples: np.ndarray,
    chosen_idx: int
) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Filter (w,m) samples to those for which the chosen index is
    consistent with being the best under that parameter.
    '''
    if param_W.shape != param_M.shape:
        raise ValueError("param_W and param_M must have same shape")
    K, d = param_W.shape
    keep_mask = np.zeros(K, dtype=bool)
    for k in range(K):
        w = param_W[k]
        m = param_M[k]
        util = PowerUtility(w=w, m=m)
        vals = util(tuples)
        if chosen_idx == int(np.argmax(vals)):
            keep_mask[k] = True
    return param_W[keep_mask], param_M[keep_mask]


def select_query_group_max_disagreement(
    X: np.ndarray,
    param_W: np.ndarray,
    param_M: np.ndarray,
    group_size: int,
    rng: np.random.Generator
) -> np.ndarray:
    '''
    Heuristic query selection:
      - sample candidate groups of 'group_size' tuples,
      - pick the one with maximum disagreement across parameters
        about which index is best.
    '''
    n = X.shape[0]
    if n <= group_size:
        return np.arange(n, dtype=int)
    # number of candidate groups to try
    num_groups = min(20, max(1, n // group_size))
    best_group = None
    best_disagreement = -1.0
    for _ in range(num_groups):
        idx = rng.choice(n, size=group_size, replace=False)
        tuples = X[idx]
        # for each parameter, compute best index in group
        best_indices = []
        for k in range(param_W.shape[0]):
            w = param_W[k]
            m = param_M[k]
            util = PowerUtility(w=w, m=m)
            vals = util(tuples)
            best_indices.append(int(np.argmax(vals)))
        # measure disagreement = number of distinct best indices
        if best_indices:
            num_distinct = len(set(best_indices))
        else:
            num_distinct = 0
        if num_distinct > best_disagreement:
            best_disagreement = num_distinct
            best_group = idx
    if best_group is None:
        best_group = rng.choice(n, size=group_size, replace=False)
    return np.asarray(best_group, dtype=int)

@dataclass
class NonlinearIODMCResult:
    questions_used: int
    num_params_remaining: int
    output_indices: np.ndarray


def iod_nonlinear_mc_interactive(
    X: np.ndarray,
    oracle: UserOracleNonlinear,
    eps: float,
    q_max: int,
    num_param_samples: int = 200,
    query_group_size: int = 5,
    m_low: float = 0.3,
    m_high: float = 2.0,
    seed: int = 0,
) -> NonlinearIODMCResult:
    """
    Monte Carlo IOD for nonlinear utilities.

    - Parameter prior: samples (w,m) ~ Dirichlet × Uniform, with no access
      to the true user utility.
    - Each query: show a small group of tuples, user picks the best.
      We keep only those parameter samples for which that choice is
      consistent (best in the group).
    - At the end: compute *worst-case regret* for each tuple over the
      remaining parameters, and keep all tuples whose worst-case regret
      is <= eps/(1+eps). This is the nonlinear IOD criterion.
    """
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    rng = np.random.default_rng(seed)

    # ----- initial parameter samples (prior only, no "true" anchoring) -----
    param_W, param_M = sample_power_params(
        d=d,
        num_samples=num_param_samples,
        rng=rng,
        m_low=m_low,
        m_high=m_high,
    )

    q_used = 0

    # ----- interactive phase: shrink parameter set using comparisons -----
    while q_used < q_max and param_W.shape[0] > 0:
        group_idx = select_query_group_max_disagreement(
            X, param_W, param_M,
            group_size=query_group_size,
            rng=rng,
        )
        tuples = X[group_idx]
        choice_local = oracle.choose(tuples)

        # keep only parameter samples where that choice is optimal
        param_W, param_M = update_param_samples_with_query(
            param_W, param_M, tuples, chosen_idx=choice_local
        )
        q_used += 1

        if param_W.shape[0] == 0:
            break

    # If all samples are eliminated, resample from prior (we simply didn't
    # explore enough; this keeps the algorithm defined).
    if param_W.shape[0] == 0:
        param_W, param_M = sample_power_params(
            d=d,
            num_samples=num_param_samples,
            rng=rng,
            m_low=m_low,
            m_high=m_high,
        )

    K = param_W.shape[0]

    # ----- final stage: worst-case regret filter (IOD output) -----
    # utilities matrix (K, n)
    utils = np.empty((K, n), dtype=float)
    for k in range(K):
        util_k = PowerUtility(w=param_W[k], m=param_M[k])
        utils[k, :] = util_k(X)

    best_vals = utils.max(axis=1)
    best_vals = np.maximum(best_vals, 1e-12)  # avoid division by 0

    regrets = (best_vals[:, None] - utils) / best_vals[:, None]
    worst_regret = regrets.max(axis=0)  # shape (n,)

    # IOD threshold: f(p*) / f(p) <= 1 + eps  <=>  regret <= eps/(1+eps)
    regret_thr = eps / (1.0 + eps)

    candidate_idx = np.where(worst_regret <= regret_thr)[0]

    # If nothing satisfies the threshold, fall back to the k lowest-regret
    # tuples to avoid empty output (purely a safety valve).
    if candidate_idx.size == 0:
        order = np.argsort(worst_regret)  # ascending (smaller regret is better)
        k_keep = min(20, n)
        candidate_idx = order[:k_keep]

    return NonlinearIODMCResult(
        questions_used=q_used,
        num_params_remaining=param_W.shape[0],
        output_indices=candidate_idx,
    )


# ==========================
# 6. Single-experiment comparison with alpha-error + question counts
# ==========================

def run_nonlinear_comparison(
    X: np.ndarray,
    true_utility: PowerUtility,
    eps: float = 0.05,
    topk_size: int = 20,
    s_squeeze: int = 5,
    q_squeeze: int = 20,        # question budget for Squeeze-u
    s_real: int = 5,
    T_real: int = 20,
    q_minR: int = 10,           # question budget for MinR
    q_minD: int = 10,           # question budget for MinD
    num_samples_for_R: int = 200,
    q_iod_budget: int = 0,      # question budget for IOD
    iod_param_samples: int = 200,
    iod_group_size: int = 5,
    seed: int = 42
) -> pd.DataFrame:
    '''
    Run one experiment (fixed X, true_utility, eps, question budgets)
    and return a DataFrame with metrics for all algorithms.
    '''
    rng = np.random.default_rng(seed)
    n, d = X.shape

    # ground truth nonlinear indistinguishability set
    t0 = time.perf_counter()
    I_idx = nonlinear_IQ_indices(X, true_utility, eps)
    t_I = time.perf_counter() - t0
    I_set = set(I_idx.tolist())
    I_size = len(I_idx)

    def tp_fp_fn_prec_rec(S_idx: np.ndarray):
        S_idx = np.asarray(S_idx, dtype=int)
        S_set = set(S_idx.tolist())
        tp = len(I_set.intersection(S_set))
        fp = len(S_set - I_set)
        fn = I_size - tp
        size = len(S_idx)
        precision = tp / size if size > 0 else 0.0
        recall = tp / I_size if I_size > 0 else 1.0
        return size, tp, fp, fn, precision, recall

    def alpha_error(fp: int, fn: int) -> float:
        '''
        alpha-error = (FP + FN) / |I|.
        0 is best (S = I). Larger is worse.
        '''
        if I_size == 0:
            return 0.0
        return (fp + fn) / I_size

    rows = []

    # linear surrogate using only coefficients (no exponents)
    lin_util = LinearUtility(u=true_utility.w)

    # 1) top-k (linear surrogate)
    t0 = time.perf_counter()
    topk_idx = top_k_indices(X, lin_util, topk_size)
    t_topk = time.perf_counter() - t0
    r_topk = regret_ratio_nonlinear(X, topk_idx, true_utility)
    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(topk_idx)
    rows.append(dict(
        Algorithm="top_k_linear",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_topk,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=0,
        Runtime_sec=t_topk
    ))

    # 2) skyline
    t0 = time.perf_counter()
    sk_idx = skyline_indices(X)
    t_sky = time.perf_counter() - t0
    r_sky = regret_ratio_nonlinear(X, sk_idx, true_utility)
    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(sk_idx)
    rows.append(dict(
        Algorithm="skyline",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_sky,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=0,
        Runtime_sec=t_sky
    ))

    # 3) IOD (interactive nonlinear, Monte Carlo over (w,m))
    oracle = UserOracleNonlinear(utility=true_utility)
    t0 = time.perf_counter()
    iod_res = iod_nonlinear_mc_interactive(
        X=X,
        oracle=oracle,
        eps=eps,
        q_max=q_iod_budget,
        num_param_samples=iod_param_samples,
        query_group_size=iod_group_size,
        seed=seed,
    )
    t_iod = time.perf_counter() - t0
    iod_idx = iod_res.output_indices
    r_iod = regret_ratio_nonlinear(X, iod_idx, true_utility)

    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(iod_idx)
    rows.append(dict(
        Algorithm="IOD_nonlinear_MC",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_iod,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=iod_res.questions_used,
        Runtime_sec=t_iod
    ))

    # 4) Squeeze-u (linear IQ alg, nonlinear user)
    t0 = time.perf_counter()
    sq_res = squeeze_u_linear_on_nonlinear_user(
        X, oracle, eps=eps,
        s=s_squeeze, q_max=q_squeeze
    )
    t_sq = time.perf_counter() - t0
    sq_idx = sq_res.output_indices
    r_sq = regret_ratio_nonlinear(X, sq_idx, true_utility)
    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(sq_idx)
    rows.append(dict(
        Algorithm="Squeeze_u_linear",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_sq,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=sq_res.questions_used,
        Runtime_sec=t_sq
    ))

    # 5) MinR
    t0 = time.perf_counter()
    mr_res = minR_minD_linear_on_nonlinear_user(
        X, oracle, eps=eps,
        s=s_real, T=T_real, q_max=q_minR,
        algo="MinR", num_samples_for_R=num_samples_for_R,
        seed=seed
    )
    t_mr = time.perf_counter() - t0
    mr_idx = mr_res.output_indices
    r_mr = regret_ratio_nonlinear(X, mr_idx, true_utility)
    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(mr_idx)
    rows.append(dict(
        Algorithm="MinR_linear",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_mr,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=mr_res.questions_used,
        Runtime_sec=t_mr
    ))

    # 6) MinD
    t0 = time.perf_counter()
    md_res = minR_minD_linear_on_nonlinear_user(
        X, oracle, eps=eps,
        s=s_real, T=T_real, q_max=q_minD,
        algo="MinD", num_samples_for_R=num_samples_for_R,
        seed=seed+1
    )
    t_md = time.perf_counter() - t0
    md_idx = md_res.output_indices
    r_md = regret_ratio_nonlinear(X, md_idx, true_utility)
    size, tp, fp, fn, prec, rec = tp_fp_fn_prec_rec(md_idx)
    rows.append(dict(
        Algorithm="MinD_linear",
        Size=size, I_size=I_size, TP=tp, FP=fp,
        Precision=prec, Recall=rec,
        Regret_true=r_md,
        Alpha_error=alpha_error(fp, fn),
        Questions_used=md_res.questions_used,
        Runtime_sec=t_md
    ))

    df = pd.DataFrame(rows)
    return df


# ==========================
# 7. Parameter sweep over eps, n, d, q_squeeze, q_iod
# ==========================

def run_param_sweep(
    eps_list,
    n_list,
    d_list,
    q_squeeze_list,
    iod_offsets,      # e.g. [7,10,15,20] to form d+offset
    mode: str = "anti_correlated",
    base_seed: int = 123,
    iod_param_samples: int = 200,
    iod_group_size: int = 5
) -> pd.DataFrame:
    '''
    Run a grid of experiments over:
      eps in eps_list
      n in n_list
      d in d_list
      q_squeeze in q_squeeze_list
      q_iod_budget in {d+offset for offset in iod_offsets}

    For each (n,d,eps,q_squeeze,q_iod) we generate ONE dataset X and ONE nonlinear
    utility, and run all algorithms with the chosen question budgets.
    '''
    all_rows = []
    exp_id = 0

    for d in d_list:
        for n in n_list:
            # fixed X and utility per (n,d) for comparability across eps and q
            seed_data = base_seed + d * 1000 + n
            X = generate_synthetic_data(n=n, d=d, mode=mode, seed=seed_data)
            rng = np.random.default_rng(seed_data)
            true_util = PowerUtility.random(d=d, rng=rng)

            for eps in eps_list:
                for q_squeeze in q_squeeze_list:
                    for offset in iod_offsets:
                        q_iod = d + offset

                        seed_run = base_seed + exp_id
                        df_one = run_nonlinear_comparison(
                            X, true_util, eps=eps,
                            topk_size=20,
                            s_squeeze=5, q_squeeze=q_squeeze,
                            s_real=5, T_real=10,
                            q_minR=q_squeeze,   # tie MinR/MinD budget to q_squeeze
                            q_minD=q_squeeze,
                            num_samples_for_R=100,
                            q_iod_budget=q_iod,
                            iod_param_samples=iod_param_samples,
                            iod_group_size=iod_group_size,
                            seed=seed_run
                        )
                        df_one["eps"] = eps
                        df_one["n"] = n
                        df_one["d"] = d
                        df_one["q_squeeze"] = q_squeeze
                        df_one["q_iod_budget"] = q_iod
                        df_one["exp_id"] = exp_id

                        all_rows.append(df_one)
                        exp_id += 1

    if not all_rows:
        return pd.DataFrame()
    return pd.concat(all_rows, ignore_index=True)


# ==========================
# 8. Example usage
# ==========================

if __name__ == "__main__":
    # Example: d = 5, n = 1000, eps in {0.01, 0.05},
    #          q_squeeze in {10,20,30,40},
    #          q_iod in {d+7, d+10, d+15, d+20}.
    eps_list = [0.01, 0.05]
    n_list = [1000]
    d_list = [5]
    q_squeeze_list = [10, 20, 30, 40]
    iod_offsets = [7, 10, 15, 20]

    df_all = run_param_sweep(
        eps_list=eps_list,
        n_list=n_list,
        d_list=d_list,
        q_squeeze_list=q_squeeze_list,
        iod_offsets=iod_offsets,
        mode="anti_correlated",
        base_seed=123,
        iod_param_samples=200,
        iod_group_size=5
    )

    pd.set_option("display.precision", 4)
    print(df_all.head(40).to_string(index=False))


       Algorithm  Size  I_size  TP  FP  Precision  Recall  Regret_true  Alpha_error  Questions_used  Runtime_sec  eps    n  d  q_squeeze  q_iod_budget  exp_id
    top_k_linear    20       3   2  18     0.1000  0.6667       0.0016       6.3333               0   3.9303e-03 0.01 1000  5         10            12       0
         skyline   590       3   3 587     0.0051  1.0000       0.0000     195.6667               0   2.2812e+00 0.01 1000  5         10            12       0
IOD_nonlinear_MC     2       3   0   2     0.0000  0.0000       0.0106       1.6667              12   9.4651e-01 0.01 1000  5         10            12       0
Squeeze_u_linear    89       3   3  86     0.0337  1.0000       0.0000      28.6667              10   3.6356e+00 0.01 1000  5         10            12       0
     MinR_linear   732       3   3 729     0.0041  1.0000       0.0000     243.0000              10   1.5904e+01 0.01 1000  5         10            12       0
     MinD_linear   732       3   3 729     0.0

In [ ]:
# ============================================
# 9. New sweep: fixed question budget q_fixed=10
#    for Squeeze-u, MinR/MinD, and nonlinear IOD
# ============================================

def run_sweep_fixed_q10(
    eps_list,
    d_list,
    n: int = 1000,
    q_fixed: int = 10,
    mode: str = "anti_correlated",
    base_seed: int = 777,
    iod_param_samples: int = 200,
    iod_group_size: int = 5,
) -> pd.DataFrame:
    """
    Run experiments over eps_list and d_list with:
      - q_squeeze  = q_fixed
      - q_iod      = q_fixed
      - q_minR     = q_fixed
      - q_minD     = q_fixed

    Returns a DataFrame with the usual metric columns plus
    (eps, n, d, q_squeeze, q_iod_budget, exp_id).
    """
    all_rows = []
    exp_id = 0

    for d in d_list:
        # fixed X and true utility per (n,d) so eps-comparisons share same data
        seed_data = base_seed + d * 1000 + n
        X = generate_synthetic_data(n=n, d=d, mode=mode, seed=seed_data)
        rng = np.random.default_rng(seed_data)
        true_util = PowerUtility.random(d=d, rng=rng)

        for eps in eps_list:
            seed_run = base_seed + exp_id
            df_one = run_nonlinear_comparison(
                X=X,
                true_utility=true_util,
                eps=eps,
                topk_size=20,
                s_squeeze=5,
                q_squeeze=q_fixed,      # Squeeze-u budget = q_fixed
                s_real=5,
                T_real=10,
                q_minR=q_fixed,         # MinR budget = q_fixed
                q_minD=q_fixed,         # MinD budget = q_fixed
                num_samples_for_R=100,
                q_iod_budget=q_fixed,   # IOD budget = q_fixed
                iod_param_samples=iod_param_samples,
                iod_group_size=iod_group_size,
                seed=seed_run,
            )

            df_one["eps"] = eps
            df_one["n"] = n
            df_one["d"] = d
            df_one["q_squeeze"] = q_fixed
            df_one["q_iod_budget"] = q_fixed
            df_one["exp_id"] = exp_id

            all_rows.append(df_one)
            exp_id += 1

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


# ---------- Example: run the fixed-q=10 sweep ----------

eps_list = [0.1, 0.01, 0.05]   # you can add more values here
d_list   = [5, 10, 15]         # change to whatever dimensions you want

df_q10 = run_sweep_fixed_q10(
    eps_list=eps_list,
    d_list=d_list,
    n=1000,
    q_fixed=10,
    mode="anti_correlated",
    base_seed=777,
    iod_param_samples=200,
    iod_group_size=5,
)

pd.set_option("display.precision", 4)
# show everything (all d, all eps, all algorithms)
print(df_q10.to_string(index=False))





       Algorithm  Size  I_size  TP  FP  Precision  Recall  Regret_true  Alpha_error  Questions_used  Runtime_sec  eps    n  d  q_squeeze  q_iod_budget  exp_id
    top_k_linear    20     127  20   0     1.0000  0.1575       0.0000       0.8425               0   6.4280e-05 0.10 1000  5         10            10       0
         skyline   618     127  74 544     0.1197  0.5827       0.0000       4.7008               0   2.3253e+00 0.10 1000  5         10            10       0
IOD_nonlinear_MC    82     127  82   0     1.0000  0.6457       0.0000       0.3543              10   6.6192e-01 0.10 1000  5         10            10       0
Squeeze_u_linear   201     127 127  74     0.6318  1.0000       0.0000       0.5827              10   5.0713e+00 0.10 1000  5         10            10       0
     MinR_linear  1000     127 127 873     0.1270  1.0000       0.0000       6.8740              10   1.9726e+01 0.10 1000  5         10            10       0
     MinD_linear  1000     127 127 873     0.1

In [ ]:
# only d = 10
df_d10 = df_q10[df_q10["d"] == 10].sort_values(["eps", "Algorithm"])
print(df_d10.to_string(index=False))



       Algorithm  Size  I_size  TP  FP  Precision  Recall  Regret_true  Alpha_error  Questions_used  Runtime_sec  eps    n  d  q_squeeze  q_iod_budget  exp_id
IOD_nonlinear_MC     1       3   1   0     1.0000  0.3333          0.0       0.6667              10   5.4838e-01 0.01 1000 10         10            10       4
     MinD_linear   997       3   3 994     0.0030  1.0000          0.0     331.3333              10   1.9478e+01 0.01 1000 10         10            10       4
     MinR_linear   997       3   3 994     0.0030  1.0000          0.0     331.3333              10   1.5719e+01 0.01 1000 10         10            10       4
Squeeze_u_linear   431       3   3 428     0.0070  1.0000          0.0     142.6667              10   5.0333e+00 0.01 1000 10         10            10       4
         skyline   981       3   1 980     0.0010  0.3333          0.0     327.3333               0   3.8582e+00 0.01 1000 10         10            10       4
    top_k_linear    20       3   3  17     0.1

In [ ]:
# only d = 15
df_d15 = df_q10[df_q10["d"] == 15].sort_values(["eps", "Algorithm"])
print(df_d15.to_string(index=False))

       Algorithm  Size  I_size  TP  FP  Precision  Recall  Regret_true  Alpha_error  Questions_used  Runtime_sec  eps    n  d  q_squeeze  q_iod_budget  exp_id
IOD_nonlinear_MC     4       6   2   2     0.5000  0.3333       0.0012       1.0000              10   3.4948e-01 0.01 1000 15         10            10       7
     MinD_linear  1000       6   6 994     0.0060  1.0000       0.0000     165.6667              10   1.9799e+01 0.01 1000 15         10            10       7
     MinR_linear  1000       6   6 994     0.0060  1.0000       0.0000     165.6667              10   1.9404e+01 0.01 1000 15         10            10       7
Squeeze_u_linear   791       6   6 785     0.0076  1.0000       0.0000     130.8333              10   5.0653e+00 0.01 1000 15         10            10       7
         skyline  1000       6   6 994     0.0060  1.0000       0.0000     165.6667               0   3.9200e+00 0.01 1000 15         10            10       7
    top_k_linear    20       6   6  14     0.3